# Job Salary Prediction: A Practical End-to-End Workflow

## 1. Problem Statement
Salary prediction is a useful but messy real-world problem. Job listings contain useful signals such as role, location, and experience, but they also come with duplicate postings, inconsistent salary formats, and a lot of noise.

In this notebook, the goal is to build a clean end-to-end pipeline that turns raw job-post data into a usable salary prediction model.

I want this notebook to feel simple and trustworthy:
- start from the raw data
- keep only the most useful insights from EDA
- build a small but effective feature set
- train the strongest model from earlier experiments
- evaluate it against a simple baseline

The focus here is not experimentation for its own sake. The focus is to show one clear path from raw data to a final model.


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)


def find_file(name: str) -> Path:
    candidates = [
        Path('/kaggle/input'),
        Path('/kaggle/working'),
        Path('.'),
        Path('data'),
        Path('../data'),
    ]
    for base in candidates:
        direct = base / name
        if direct.exists():
            return direct
        if base.exists():
            matches = list(base.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError(f'Could not find {name!r}. Add it as a Kaggle dataset/input or place it near the notebook.')


JOB_CSV = find_file('job.csv')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('job.csv ->', JOB_CSV)
print('output dir ->', OUTPUT_DIR.resolve())


## 2. Data Loading
I start with the raw scraped dataset and create a lightweight modeling table from it.

The cleaning in this notebook is intentionally narrow: only the steps that are needed to make the target usable and the final training set reliable are kept. That means parsing salary ranges, extracting experience values, and filtering out noisy duplicate-style postings.


In [ ]:
raw_df = pd.read_csv(JOB_CSV)
raw_df.head()


In [ ]:
def parse_salary(text: str) -> pd.Series:
    text = str(text)
    if 'competitive salary' in text.lower():
        return pd.Series([pd.NA, pd.NA, False], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])
    nums = re.findall(r'[\d,]+', text)
    nums = [int(n.replace(',', '')) for n in nums]
    if not nums:
        return pd.Series([pd.NA, pd.NA, False], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])
    if len(nums) == 1:
        return pd.Series([nums[0], nums[0], True], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])
    return pd.Series([nums[0], nums[1], True], index=['min_ctc', 'max_ctc', 'is_salary_disclosed'])


salary_parts = raw_df['ctc'].apply(parse_salary)
df = pd.concat([raw_df.copy(), salary_parts], axis=1)
df['avg_ctc'] = pd.to_numeric(df['min_ctc'], errors='coerce').add(pd.to_numeric(df['max_ctc'], errors='coerce')).div(2)
df['min_exp'] = pd.to_numeric(df['experience'].str.extract(r'(\d+)')[0], errors='coerce')
df['max_exp'] = pd.to_numeric(df['experience'].str.extract(r'(?:\d+)-(\d+)')[0], errors='coerce').fillna(df['min_exp'])
df['is_noisy_duplicate'] = df.duplicated(subset=['job_title', 'company_name', 'location'], keep='first')

model_df = df.loc[(~df['is_noisy_duplicate']) & (df['avg_ctc'].notna())].copy()

summary = pd.DataFrame({
    'metric': ['Raw rows', 'Rows after dedup + target filtering', 'Salary disclosure rate', 'Median salary'],
    'value': [
        len(raw_df),
        len(model_df),
        round(float(df['is_salary_disclosed'].mean()), 3),
        round(float(model_df['avg_ctc'].median()), 2),
    ],
})
summary


## 3. Key EDA Insights
The full exploratory analysis was done earlier, so this section keeps only the observations that directly shape the final model.

**The most important takeaways were:**
- Salary is strongly right-skewed, so simple averages do not tell the full story.
- Experience has a clearer relationship with salary than posting recency.
- Job title and location contain strong market signal.
- Duplicate-style listings and undisclosed salaries can easily distort the training data if they are left untreated.

These points matter because they explain why the final model is built around a small set of stable predictors instead of a long list of noisy features.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(model_df['avg_ctc'], bins=40, ax=axes[0], color='#4c78a8')
axes[0].set_title('Salary Distribution')
axes[0].set_xlabel('Average CTC')

corr_df = model_df[['min_exp', 'max_exp', 'avg_ctc']].corr(numeric_only=True)
sns.heatmap(corr_df, annot=True, cmap='Blues', fmt='.2f', ax=axes[1])
axes[1].set_title('Core Numeric Relationships')

plt.tight_layout()


## 4. Feature Engineering
The final feature set is intentionally small. That keeps the pipeline easier to understand and reduces the chance of carrying unstable features into the final model.

### Target
- `avg_ctc`

### Features
- `min_exp`
- `max_exp`
- `job_title`
- `location`

### Preprocessing
- numeric columns: median imputation
- categorical columns: most-frequent imputation + one-hot encoding

One deliberate choice here is to leave out posting recency. In earlier analysis it showed weak and inconsistent signal, so including it made the model less stable without adding much predictive value.


In [ ]:
target = 'avg_ctc'
features = ['min_exp', 'max_exp', 'job_title', 'location']

train_df, test_df = train_test_split(model_df, test_size=0.2, random_state=42)

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

numeric_features = ['min_exp', 'max_exp']
categorical_features = ['job_title', 'location']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)


## 5. Model (Best Performing)
From the earlier modeling notebook, tree-based methods clearly outperformed the simple baseline. Among them, XGBoost gave the strongest overall result, so that is the model I carry forward here.

To keep this notebook readable, I use the tuned XGBoost configuration directly instead of re-running a long experimental search. I still keep a simple linear regression baseline in the evaluation step so the improvement stays easy to see.


In [ ]:
baseline_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', LinearRegression()),
])

final_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        n_estimators=200,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=1,
    )),
])

baseline_model.fit(X_train, y_train)
final_model.fit(X_train, y_train)


## 6. Evaluation
I evaluate the final model with two metrics:
- **MAE**: the average absolute error in salary units, which is the easiest metric to explain in business terms
- **R2**: how much of the variation in salary the model explains overall

The main question is straightforward: does the final model improve enough over the linear baseline to justify the extra complexity?


In [ ]:
baseline_pred = baseline_model.predict(X_test)
final_pred = final_model.predict(X_test)

results = pd.DataFrame([
    {
        'model': 'Linear Regression baseline',
        'mae': mean_absolute_error(y_test, baseline_pred),
        'r2': r2_score(y_test, baseline_pred),
    },
    {
        'model': 'Tuned XGBoost final model',
        'mae': mean_absolute_error(y_test, final_pred),
        'r2': r2_score(y_test, final_pred),
    },
]).sort_values('mae').reset_index(drop=True)

results.assign(mae=results['mae'].round(2), r2=results['r2'].round(4))


In [ ]:
evaluation_df = X_test.copy()
evaluation_df['actual_salary'] = y_test.values
evaluation_df['predicted_salary'] = final_pred
evaluation_df['absolute_error'] = (evaluation_df['actual_salary'] - evaluation_df['predicted_salary']).abs()

evaluation_df.sort_values('absolute_error', ascending=False).head(10)


In [ ]:
feature_names = final_model.named_steps['preprocess'].get_feature_names_out()
feature_importance = pd.Series(
    final_model.named_steps['model'].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(evaluation_df['absolute_error'], bins=40, ax=axes[0], color='#f58518')
axes[0].set_title('Absolute Error Distribution')
axes[0].set_xlabel('Absolute Error')

feature_importance.head(12).sort_values().plot(kind='barh', ax=axes[1], color='#54a24b')
axes[1].set_title('Top 12 Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()


## 7. Conclusion
This notebook is the cleaned-up version of the whole project story.

It starts from raw job listings, keeps only the most important analytical context, builds a small and stable feature set, and ends with the strongest model from the earlier comparison work.

### Final takeaway
The tuned XGBoost model improves clearly over the linear baseline while still fitting into a compact pipeline that is easy to follow.

### Why I like this final version
- it feels like one project instead of a stack of experiments
- it keeps the learning journey visible without dragging all the clutter along
- it is simple enough to read in one pass and solid enough to reuse
